# Gemini で大量のテキストをまとめて処理する (Batch API)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/15_llm_api/gemini_batch.ipynb)

NLP クックブック 15-06 ／ LLM の API（Gemini と Claude）

> API キーが必要（Colab 左の 🔑 シークレットに登録）

数百〜数千件のテキストを LLM で処理したい。1 件ずつリクエストを送る代わりに、まとめて送って、後から結果を受け取りたい。

In [ ]:
!pip install -q google-genai

In [ ]:
import time
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-flash-latest"

reviews = ["画面がきれいで操作も軽快。買ってよかった。", "届いてすぐ壊れた。サポートの対応も遅い。", "値段相応。可もなく不可もなく。",
           "思っていたより小さかったが、使いやすい。", "説明書が分かりにくく、設定に苦労した。"]
requests = [{"contents": [{"role": "user", "parts": [{"text": f"次のレビューを「ポジティブ」「ネガティブ」「中立」のどれかに分類し、ラベルだけを出力してください。\n\n{r}"}]}],
             "metadata": {"key": str(i)}}                  # 結果と入力を対応付けるための番号
            for i, r in enumerate(reviews)]
job = client.batches.create(model=MODEL, src=requests, config={"display_name": "review-sentiment"})
print(job.name, job.state.name)          # ジョブ名を控えておけば、セッションが切れても後から結果を取得できる

In [ ]:
# 完了を待つ (数分〜最大 24 時間)
while True:
    job = client.batches.get(name=job.name)
    if job.state.name in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"):
        break
    print(job.state.name); time.sleep(30)
print(job.state.name)

In [ ]:
for review, r in zip(reviews, job.dest.inlined_responses):
    print((r.response.text.strip() if r.response else f"エラー: {r.error}"), "|", review)

- バッチは非同期で処理される代わりに、料金が通常の約半額になる。急がない大量の処理（データのラベル付け、評価、翻訳など）に向く。
- バッチ処理は無料枠では使えないことがある（有料枠の機能として提供されている時期がある）。使えない場合は、1 件ずつ `generate_content` を呼び、レート制限に当たったら待って再試行する。
- 数千件以上なら、リクエストを JSON Lines のファイルにしてアップロードする方法（`client.files.upload` で上げたファイルを `src` に渡す）を使う。
- 結果の順序に頼らず、`metadata` に付けた番号で入力と対応付けるのが安全。